# Fine-tune DistilBERT on IMDB

This notebook fine-tunes `distilbert-base-uncased` for binary sentiment classification using Hugging Face's IMDB dataset. It uses a smaller sample by default so it can be tested on a laptop; set `FAST_RUN = False` to use the complete dataset. A GPU is recommended but not required.

In [ ]:
# The project already declares these dependencies. If needed, uncomment this line:
# %pip install -q "datasets==3.6.0" "transformers>=4.56.2" "torch>=2.8.0" "scikit-learn>=1.7.2"

In [1]:
from pathlib import Path
import random

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = Path("imdb-distilbert-finetuned")
MAX_LENGTH = 256
FAST_RUN = True  # Change to False to use all 25,000 training reviews.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device_name = f"CUDA ({torch.cuda.get_device_name(0)})"
elif torch.backends.mps.is_available():
    device_name = "Apple Silicon (MPS)"
else:
    device_name = "CPU"
print(f"Training device: {device_name}")

Training device: Apple Silicon (MPS)


## 1. Load and split the dataset

We hold out 10% of the training data for validation and reserve the official test split for final evaluation. Labels are `0` (negative) and `1` (positive).

In [3]:
raw_imdb = load_dataset("stanfordnlp/imdb")
train_valid = raw_imdb["train"].train_test_split(
    test_size=0.10, seed=SEED, stratify_by_column="label"
)
train_dataset = train_valid["train"]
validation_dataset = train_valid["test"]
test_dataset = raw_imdb["test"]

if FAST_RUN:
    # Shuffle first so each smaller sample remains representative.
    train_dataset = train_dataset.shuffle(seed=SEED).select(range(5_000))
    validation_dataset = validation_dataset.shuffle(seed=SEED).select(range(1_000))
    test_dataset = test_dataset.shuffle(seed=SEED).select(range(1_000))

print(f"Train: {len(train_dataset):,}")
print(f"Validation: {len(validation_dataset):,}")
print(f"Test: {len(test_dataset):,}")
train_dataset[0]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train: 5,000
Validation: 1,000
Test: 1,000


{'text': 'E! TV is a great channel and Talk Soup is so funny,in a flash you can view the episodes change. We want more funny writings by the best writer ever Stan Evans.. The patron Saint of the mindless masses... He is a truly talented, gifted writer, actor, comic, producer,director, and creative consultant.Anna Nicole loved him , but he was not a $$$$Billionaire so he left him for a Billionaire. Many super stars wanted to make films with the actor Stan Evans, who has a "Humphrey Bogart" {Clark Gable}acting style. He should make many more movies. Maybe with Stephen Spielberg, or perhaps many other talented producers.We wish him a moment of FAME with a great fortune to gain. Has he produced any mock-U-dramas? or perhaps any docudrama??? A project about Bernie Madhoff would be a great TV movie written by STAN EVANS. How many screenplays has he written?? Is he under $$$$$$$$$$$$billion contract with Disney?? He should earn more than $50 Million... He could also write a TV movie about the

## 2. Tokenize the reviews

Reviews are truncated to 256 tokens. Dynamic padding pads each batch only to its longest example, reducing wasted computation.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_train = train_dataset.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)
tokenized_validation = validation_dataset.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)
tokenized_test = test_dataset.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_train

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 5000
})

## 3. Create the classification model and metrics

In [5]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {label: index for index, label in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4. Fine-tune

The best checkpoint is selected by validation accuracy. CUDA uses mixed precision; MPS and CPU use standard precision for compatibility.

In [6]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
trainer.log_metrics("train", train_result.metrics)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

## 5. Evaluate on the held-out test set

In [ ]:
test_metrics = trainer.evaluate(tokenized_test, metric_key_prefix="test")
trainer.log_metrics("test", test_metrics)
test_metrics

## 6. Save the model and run inference

The model and tokenizer are saved in `imdb-distilbert-finetuned/final`, ready to load later with `from_pretrained`.

In [ ]:
FINAL_MODEL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
print(f"Saved model to: {FINAL_MODEL_DIR.resolve()}")

In [ ]:
def predict_sentiment(text):
    model.eval()
    model_device = next(model.parameters()).device
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    ).to(model_device)
    with torch.inference_mode():
        probabilities = torch.softmax(model(**inputs).logits, dim=-1)[0]
    predicted_id = int(probabilities.argmax().item())
    return {
        "label": model.config.id2label[predicted_id],
        "confidence": float(probabilities[predicted_id].item()),
    }

examples = [
    "A smart, moving film with excellent performances.",
    "The plot was tedious and I regretted watching it.",
]
for review in examples:
    print(review, "->", predict_sentiment(review))